# ⚙️ NorthAir Intelligence — 03. Data Preprocessing
## VIZ-A-THON 2026: Time Zone Conversion, Reshaping & Analytical Enrichment

### Objectives:
1. **Timestamp Processing**: Convert UTC timestamps to Indian Standard Time (IST = UTC + 05:30) and engineer temporal features (`Date`, `Year`, `Month`, `Month_Name`, `Day`, `Hour`, `Day_of_Week`).
2. **Historical AQI Transformation**: Convert 24 hourly wide columns (`00:00:00` to `23:00:00`) to long format via Pandas `melt`, verifying complete non-null AQI observation parity.
3. **Station Metadata Consolidation**: Enrich station inventory with coverage statistics and presence indicators.
4. **Artifact Persistence**: Save analysis-ready datasets in both CSV and Parquet formats in `data/processed/`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 35)

sys.path.insert(0, str(Path("../src").resolve()))
from northair.data_loader import DataLoader
from northair.preprocessing import (
    convert_utc_to_ist,
    transform_aqi_history_to_long,
    build_station_metadata,
    save_processed_datasets,
)
from northair.validation import (
    validate_timestamp_conversion,
    validate_aqi_melting,
    validate_processed_outputs,
)

loader = DataLoader(raw_dir=Path("../data/raw"))
stations_df = loader.load_stations()
aqi_hist_df = loader.load_aqi_history()
aq_2025_df = loader.load_air_quality_hourly_2025()
print("Raw datasets loaded.")


### 1. Task 3: Timestamp Processing (UTC → IST)
The raw 2025 timestamps are recorded in UTC. Local air pollution patterns (e.g. rush-hour vehicular peaks, diurnal boundary layer compression, agricultural open-fire timing) require Indian Standard Time (UTC + 05:30).


In [ ]:
aq_2025_ist, ts_anomalies = convert_utc_to_ist(aq_2025_df, timestamp_col="Timestamp")

# Validate conversion
val_ts = validate_timestamp_conversion(aq_2025_df, aq_2025_ist)
print(f"Timestamp Validation: {val_ts['offset_verified']} verified.")
print(f"Missing timestamps: {ts_anomalies['missing_timestamps']}, Invalid timestamps: {ts_anomalies['invalid_timestamps']}\n")

display(aq_2025_ist[['Station ID', 'Timestamp', 'Timestamp_IST', 'Date', 'Month_Name', 'Hour', 'Day_of_Week', 'PM2.5 (µg/m³)']].head(5))


### 2. Diurnal Pollution Dynamics in Local IST
With the IST conversion completed, we can observe the characteristic diurnal peak of PM2.5 in Indian Standard Time (morning and evening traffic / inversions).


In [ ]:
# Diurnal PM2.5 Profile in IST
hourly_pm25 = aq_2025_ist.groupby('Hour')['PM2.5 (µg/m³)'].median().reset_index()

plt.figure(figsize=(10, 4))
sns.lineplot(data=hourly_pm25, x='Hour', y='PM2.5 (µg/m³)', marker='o', color='purple', linewidth=2.5)
plt.title("Median Diurnal PM2.5 Profile Across North India (IST)", fontsize=13, fontweight='bold')
plt.xlabel("Hour of Day (IST: 0 to 23)")
plt.ylabel("Median PM2.5 (µg/m³)")
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


### 3. Task 4: Historical AQI Transformation (Wide → Long)
We reshape `aqi_history_2017_2023.csv` from wide daily columns (`00:00:00` - `23:00:00`) into a continuous time series table.


In [ ]:
aqi_long, melt_metrics = transform_aqi_history_to_long(aqi_hist_df)

print(f"Wide rows: {melt_metrics['wide_rows']:,}")
print(f"Long rows: {melt_metrics['long_rows']:,}")
print(f"Wide non-null AQI observations: {melt_metrics['wide_non_null_aqi']:,}")
print(f"Long non-null AQI observations: {melt_metrics['long_non_null_aqi']:,}")
print(f"Parity verified: {melt_metrics['parity_verified']}\n")

display(aqi_long.head(6))


### 4. Task 5: Station Metadata Consolidation
We generate an authoritative station registry enriched with operational dates and observation volume.


In [ ]:
station_meta = build_station_metadata(
    stations_df=stations_df,
    air_quality_2025_df=aq_2025_df,
    aqi_history_df=aqi_hist_df,
)

display(station_meta.head(10))


### 5. Task 6: Export Processed Datasets (CSV & Parquet)
We persist all transformed datasets to `data/processed/`.


In [ ]:
output_summary = save_processed_datasets(
    datasets={
        "air_quality_2025_ist": aq_2025_ist,
        "aqi_history_long": aqi_long,
        "station_metadata": station_meta,
    },
    output_dir=Path("../data/processed"),
    save_parquet=True,
)

val_outputs = validate_processed_outputs(output_dir=Path("../data/processed"))
print(f"Validation Status: {'PASSED' if val_outputs['passed'] else 'FAILED'}\n")
for fname, info in val_outputs['files'].items():
    print(f"  • {fname:<28}: {info['size_mb']:>6.2f} MB")


### Summary of Completed Preprocessing:
1. Converted UTC timestamps to Indian Standard Time (IST = UTC + 05:30) with 8 analytical temporal columns.
2. Reshaped 2017-2023 historical AQI into tidy 2.46M row time series with 100% non-null observation parity.
3. Built consolidated station metadata catalog with presence indicators.
4. Saved dual CSV and columnar Parquet formats in `data/processed/`.
